# Day 2 — SQL Analysis: 8 Business Questions

## Objective
Answer 8 business questions about SLA performance using SQL queries
on the clean dataset built in Day 1. All queries run against an
in-memory SQLite database loaded from `orders_clean.csv`.

## Why SQL, not pandas?
SQL is declarative — it describes *what* you want, not *how* to get it.
This makes business questions read more naturally as queries, and it is
the primary skill tested in junior analyst interviews and take-home tasks.
SQLite runs entirely in Python with no server setup required.


In [4]:
import pandas as pd
import sqlite3

# Load the clean dataset produced by Day 1
df = pd.read_csv('../data/clean/orders_clean.csv')
print(f"Loaded: {df.shape[0]:,} rows | {df.shape[1]} columns")

# Create an in-memory SQLite database and load the dataframe into it
# ':memory:' means the database exists only for this session
# no files written, no cleanup needed
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')
print("SQLite database created — table: 'orders'")

# Helper function: run a SQL query and return a clean dataframe
# Using this wrapper keeps query cells clean and consistent
def sql(query):
    return pd.read_sql_query(query, conn)

# Confirm the table loaded correctly
result = sql("SELECT COUNT(*) AS total_rows FROM orders")
print(f"Rows in SQL table: {result['total_rows'][0]:,}")

Loaded: 96,470 rows | 28 columns
SQLite database created — table: 'orders'
Rows in SQL table: 96,470


## Q1 — Overall SLA Performance Summary

**Business question:** What is the headline SLA performance of the platform?

**Why this comes first:** Every subsequent number is only meaningful relative
to this baseline. A 6.8% breach rate means nothing without knowing total volume,
average delay severity, and how far early/late orders skew.

In [5]:
q1 = sql('''
    SELECT
        COUNT(*)                                    AS total_orders,
        SUM(sla_breached)                           AS breached_orders,
        ROUND(AVG(sla_breached) * 100, 2)           AS breach_rate_pct,
        ROUND(AVG(days_vs_sla), 1)                  AS avg_days_vs_sla,
        ROUND(AVG(actual_days), 1)                  AS avg_actual_days,
        ROUND(AVG(promised_days), 1)                AS avg_promised_days,
        ROUND(MIN(days_vs_sla), 0)                  AS earliest_days,
        ROUND(MAX(days_vs_sla), 0)                  AS latest_days
    FROM orders
''')

print("=== Q1: Overall SLA Performance ===")
print(q1.to_string(index=False))

print("\n--- Plain English Summary ---")
print(f"Total orders analysed:    {q1['total_orders'][0]:,}")
print(f"Orders breached SLA:      {q1['breached_orders'][0]:,}")
print(f"Overall breach rate:      {q1['breach_rate_pct'][0]}%")
print(f"Average days vs SLA:      {q1['avg_days_vs_sla'][0]} days "
      f"({'early' if q1['avg_days_vs_sla'][0] < 0 else 'late'})")
print(f"Average promised window:  {q1['avg_promised_days'][0]} days")
print(f"Average actual delivery:  {q1['avg_actual_days'][0]} days")
print(f"Earliest delivery:        {abs(q1['earliest_days'][0]):.0f} days early")
print(f"Latest delivery:          {q1['latest_days'][0]:.0f} days late")

=== Q1: Overall SLA Performance ===
 total_orders  breached_orders  breach_rate_pct  avg_days_vs_sla  avg_actual_days  avg_promised_days  earliest_days  latest_days
        96470             6534             6.77            -11.9             12.1               23.4         -147.0        188.0

--- Plain English Summary ---
Total orders analysed:    96,470
Orders breached SLA:      6,534
Overall breach rate:      6.77%
Average days vs SLA:      -11.9 days (early)
Average promised window:  23.4 days
Average actual delivery:  12.1 days
Earliest delivery:        147 days early
Latest delivery:          188 days late


## Q2 — Breach Severity Breakdown

**Business question:** Of the 6.77% of orders that breached SLA,
how bad were the breaches?

**Why this comes second:** The headline breach rate (6.77%) needs context.
A platform where all breaches are 1 day late is very different from one
where breaches run 2+ weeks late. This query separates cosmetic breaches
from genuinely damaging ones.

**Note on outliers:** Q1 revealed a range of -147 to +188 days vs SLA.
These extreme values are likely data quality issues but are retained
since they don't distort the breach rate itself, only the averages.
They are visible in the severity breakdown below.

In [6]:
q2 = sql('''
    SELECT
        breach_severity,
        COUNT(*)                                        AS order_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_total,
        ROUND(AVG(days_vs_sla), 1)                     AS avg_days_vs_sla,
        ROUND(MIN(days_vs_sla), 0)                     AS min_days,
        ROUND(MAX(days_vs_sla), 0)                     AS max_days
    FROM orders
    GROUP BY breach_severity
    ORDER BY
        CASE breach_severity
            WHEN 'On Time / Early'       THEN 1
            WHEN 'Minor (1-3 days late)' THEN 2
            WHEN 'Moderate (4-7 days late)' THEN 3
            WHEN 'Severe (7+ days late)' THEN 4
        END
''')

print("=== Q2: Breach Severity Breakdown ===")
print(q2.to_string(index=False))

# Pull out the on-time row for context
on_time = q2[q2['breach_severity'] == 'On Time / Early']['order_count'].values[0]
severe   = q2[q2['breach_severity'] == 'Severe (7+ days late)']['order_count'].values[0]

print("\n--- Plain English Summary ---")
print(f"On time or early:     {on_time:,} orders ({on_time/96470*100:.1f}%)")
print(f"Severe breaches only: {severe:,} orders ({severe/96470*100:.1f}% of all orders)")
print(f"Key insight: of the 6.77% that breached, check what % were severe vs minor")

=== Q2: Breach Severity Breakdown ===
         breach_severity  order_count  pct_of_total  avg_days_vs_sla  min_days  max_days
         On Time / Early        89936         93.23            -13.5    -147.0       0.0
   Minor (1-3 days late)         1870          1.94              1.8       1.0       3.0
Moderate (4-7 days late)         1802          1.87              5.5       4.0       7.0
   Severe (7+ days late)         2862          2.97             19.6       8.0     188.0

--- Plain English Summary ---
On time or early:     89,936 orders (93.2%)
Severe breaches only: 2,862 orders (3.0% of all orders)
Key insight: of the 6.77% that breached, check what % were severe vs minor


## Q3 — Monthly Breach Rate Trend

**Business question:** Is SLA performance improving or deteriorating over time?

**Why this matters:** A static breach rate hides directional movement.
A platform improving from 10% to 6.77% tells a different story than one
that has been stuck at 6.77% for 2 years — or one that was 3% and is getting worse.
The trend determines urgency of the recommendations.

**Note on ordering:** Month names are ordered chronologically using
MIN(order_purchase_timestamp) as the sort key the earliest actual
purchase date in each month-name group. This avoids alphabetical sorting
without needing a numeric month column.

In [7]:
q3 = sql('''
    SELECT
        STRFTIME('%Y-%m', order_purchase_timestamp)     AS year_month,
        COUNT(*)                                        AS total_orders,
        SUM(sla_breached)                               AS breached_orders,
        ROUND(AVG(sla_breached) * 100, 2)               AS breach_rate_pct,
        ROUND(AVG(days_vs_sla), 1)                      AS avg_days_vs_sla
    FROM orders
    GROUP BY year_month
    ORDER BY year_month
''')

print("=== Q3: Monthly Breach Rate Trend ===")
print(q3.to_string(index=False))

# Highlight worst and best months
worst_month = q3.loc[q3['breach_rate_pct'].idxmax()]
best_month  = q3.loc[q3['breach_rate_pct'].idxmin()]

print("\n--- Plain English Summary ---")
print(f"Months analysed:    {len(q3)}")
print(f"Worst month:        {worst_month['year_month']} "
      f"— {worst_month['breach_rate_pct']}% breach rate")
print(f"Best month:         {best_month['year_month']} "
      f"— {best_month['breach_rate_pct']}% breach rate")
print(f"Overall direction:  compare first 3 months avg vs last 3 months avg below")

first_3 = q3.head(3)['breach_rate_pct'].mean().round(2)
last_3  = q3.tail(3)['breach_rate_pct'].mean().round(2)
direction = "improving" if last_3 < first_3 else "deteriorating"
print(f"First 3 months avg: {first_3}%")
print(f"Last 3 months avg:  {last_3}%")
print(f"Trend:              {direction}")

=== Q3: Monthly Breach Rate Trend ===
year_month  total_orders  breached_orders  breach_rate_pct  avg_days_vs_sla
   2016-09             1                1           100.00             36.0
   2016-10           265                2             0.75            -36.7
   2016-12             1                0             0.00            -22.0
   2017-01           750               22             2.93            -27.4
   2017-02          1653               49             2.96            -19.2
   2017-03          2546              116             4.56            -12.3
   2017-04          2303              151             6.56            -13.0
   2017-05          3545              106             2.99            -13.5
   2017-06          3135               95             3.03            -12.6
   2017-07          3872              108             2.79            -12.5
   2017-08          4193              122             2.91            -13.1
   2017-09          4150              182         